In [1]:
import pandas as pd
ratings = pd.read_csv('data/ratings_small.csv')

In [2]:
min = ratings['rating'].min()
max = ratings['rating'].max()
min, max

(0.5, 5.0)

In [3]:
from surprise import Reader, Dataset, SVD

In [4]:
reader = Reader(rating_scale=(min, max))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

In [5]:
svd = SVD(random_state=0)
trainset = data.build_full_trainset()
svd.fit(trainset)

In [6]:
ratings.head(2)

,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179


In [7]:
svd.predict(1, 31)

Prediction(uid=1, iid=31, r_ui=None, est=2.4162799702909346, details={'was_impossible': False})

In [18]:
uid = 9
mid = 42
seen_movies = ratings[ratings['userId']==uid]['movieId']

if seen_movies[seen_movies==mid].count() == 0:
  print(f'사용자:{uid}는 영화{mid} 평점없음')
  print(svd.predict(uid, mid))

사용자:9는 영화42 평점없음
user: 9          item: 42         r_ui = None   est = 2.94   {'was_impossible': False}


In [25]:
#평점매긴 영화목록
uid=9
filt = ratings['userId'] == uid
seen_movies = list(ratings[filt]['movieId'])
len(seen_movies)

45

In [27]:
#평점매긴 전체 영화목록
total_movies = ratings['movieId'].drop_duplicates().tolist()
len(total_movies)

9066

In [28]:
#추천 영화목록
import numpy as np
unseen_movies = np.setdiff1d(total_movies, seen_movies)
len(unseen_movies)

9021

In [33]:
def get_unseen_movies(ratings, uid):
  filt = ratings['userId'] == uid
  seen_movies = list(ratings[filt]['movieId'])
  total_movies = ratings['movieId'].drop_duplicates().tolist()
  unseen_movies = np.setdiff1d(total_movies, seen_movies)
  print(f'사용자 아이디:{uid} 평점매긴 영화수:{len(seen_movies)} 추천대상 영화수:{len(unseen_movies)}')
  return unseen_movies

In [34]:
unseen_movies = get_unseen_movies(ratings, 9)

사용자 아이디:9 평점매긴 영화수:45 추천대상 영화수:9021


In [37]:
predict = [svd.predict(uid, mid) for mid in unseen_movies]
predict

[Prediction(uid=9, iid=2, r_ui=None, est=3.3022661202336026, details={'was_impossible': False}),
 Prediction(uid=9, iid=3, r_ui=None, est=3.1564257268610603, details={'was_impossible': False}),
 Prediction(uid=9, iid=4, r_ui=None, est=2.700195831006973, details={'was_impossible': False}),
 Prediction(uid=9, iid=5, r_ui=None, est=3.114698903163602, details={'was_impossible': False}),
 Prediction(uid=9, iid=6, r_ui=None, est=3.904805942864381, details={'was_impossible': False}),
 Prediction(uid=9, iid=7, r_ui=None, est=3.059621203539996, details={'was_impossible': False}),
 Prediction(uid=9, iid=8, r_ui=None, est=3.644808013555, details={'was_impossible': False}),
 Prediction(uid=9, iid=9, r_ui=None, est=2.842665999241858, details={'was_impossible': False}),
 Prediction(uid=9, iid=10, r_ui=None, est=3.399962356260424, details={'was_impossible': False}),
 Prediction(uid=9, iid=11, r_ui=None, est=3.479578700970142, details={'was_impossible': False}),
 Prediction(uid=9, iid=12, r_ui=None, e

In [39]:
predict.sort(key=lambda pre:pre.est, reverse=True)
top_predict = predict[:5]
top_predict

[Prediction(uid=9, iid=858, r_ui=None, est=4.542866877335705, details={'was_impossible': False}),
 Prediction(uid=9, iid=912, r_ui=None, est=4.484090707192216, details={'was_impossible': False}),
 Prediction(uid=9, iid=4993, r_ui=None, est=4.471004680156093, details={'was_impossible': False}),
 Prediction(uid=9, iid=926, r_ui=None, est=4.427937145395248, details={'was_impossible': False}),
 Prediction(uid=9, iid=745, r_ui=None, est=4.41983077978538, details={'was_impossible': False})]

In [40]:
top_movies = [(pred.iid, pred.est) for pred in top_predict]
top_movies

[(858, 4.542866877335705),
 (912, 4.484090707192216),
 (4993, 4.471004680156093),
 (926, 4.427937145395248),
 (745, 4.41983077978538)]

In [41]:
def get_recommendations(svd, uid, unseen_movies, top_n):
  predict = [svd.predict(uid, mid) for mid in unseen_movies]
  predict.sort(key=lambda pre:pre.est, reverse=True)
  top_predict = predict[:top_n]
  top_movies = [(pred.iid, pred.est) for pred in top_predict]
  return top_movies


In [51]:
from tmdbv3api import Movie, TMDb
movie = Movie()
tmdb = TMDb()
tmdb.api_key='c668cda4cf75bf267ef2aeffa2da0341'
tmdb.language='ko-KR'

In [60]:
uid=2
top_n=5
unseen_movies = get_unseen_movies(ratings, uid)
top_movies=get_recommendations(svd, uid, unseen_movies, top_n)
print('\n-----추천 영화 리스트------')
for top in top_movies:
  mid = top[0]
  try:
    details = movie.details(mid)
    print('영화제목:' + details['title'])
  except:
    print('영화제목없음')
  print(f'영화아이디:{top[0]}, 예상평점:{top[1]}')
  print('-' * 50) 

사용자 아이디:2 평점매긴 영화수:76 추천대상 영화수:8990

-----추천 영화 리스트------
영화제목:밀리언 달러 호텔
영화아이디:318, 예상평점:4.536088707483336
--------------------------------------------------
영화제목없음
영화아이디:8132, 예상평점:4.511074801960958
--------------------------------------------------
영화제목:갤럭시 퀘스트
영화아이디:926, 예상평점:4.474115364443329
--------------------------------------------------
영화제목없음
영화아이디:1276, 예상평점:4.4659387745879355
--------------------------------------------------
영화제목:패딩턴 발 4시 50분
영화아이디:750, 예상평점:4.440999064422496
--------------------------------------------------
